# Querying Blockchain Data with SQL in Python
## Etherscan → SQLite → SQL → pandas
### What we would be learning this today:
- Creates a local blockchain.db SQLite database (real schema)
- Fetches live Ethereum data from the Etherscan API
- Runs SQL queries — basic SELECT through window functions and log decoding
- Loads results into pandas and exports to CSV
- Setup (run once in your terminal):

! pip install requests python-dotenv pandas

Create a .env file with your free API key from https://etherscan.io:

ETHERSCAN_API_KEY=YourKeyHere

In [1]:
! pip install requests python-dotenv pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Configuration & Database Schema

In [2]:
import os, sqlite3, requests, time
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

API_KEY  = os.getenv("ETHERSCAN_API_KEY")

print ("Etherscan API is loaded")

Etherscan API is loaded


In [3]:
BASE_URL = "https://api.etherscan.io/v2/api" 
DB_PATH  = "blockchain.db"
CHAIN_ID = 1   # Ethereum mainnet

CONTRACTS = {
    "USDC": "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48",
    "USDT": "0xdAC17F958D2ee523a2206206994597C13D831ec7",
    "WETH": "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2",
}

TRANSFER_TOPIC = "0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef"

def call(module, action, **params):
    resp = requests.get(BASE_URL, params={
        "chainid": CHAIN_ID, "module": module,
        "action": action,    "apikey": API_KEY,
        **params
    }, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    if data.get("status") == "0" and data.get("message") != "No transactions found":
        raise RuntimeError(f"Etherscan [{action}]: {data.get('result', data.get('message'))}")
    return data.get("result")

def wei_to_eth(v): return int(v) / 1e18
def ts_to_date(ts): return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime("%Y-%m-%d")

# Create SQLite database 
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

conn.executescript("""
    CREATE TABLE IF NOT EXISTS transactions (
        hash             TEXT PRIMARY KEY,
        block_number     INTEGER,
        block_date       TEXT,
        block_timestamp  INTEGER,
        from_address     TEXT,
        to_address       TEXT,
        value_eth        REAL,
        gas_used         INTEGER,
        gas_price_gwei   REAL,
        gas_cost_eth     REAL,
        is_error         INTEGER DEFAULT 0
    );

    CREATE TABLE IF NOT EXISTS token_transfers (
        hash             TEXT,
        log_index        INTEGER,
        block_date       TEXT,
        block_timestamp  INTEGER,
        from_address     TEXT,
        to_address       TEXT,
        contract_address TEXT,
        token_symbol     TEXT,
        token_decimals   INTEGER,
        amount           REAL,
        PRIMARY KEY (hash, log_index)
    );

    CREATE TABLE IF NOT EXISTS nft_transfers (
        hash             TEXT,
        log_index        INTEGER,
        block_date       TEXT,
        from_address     TEXT,
        to_address       TEXT,
        contract_address TEXT,
        token_name       TEXT,
        token_id         TEXT,
        PRIMARY KEY (hash, log_index)
    );

    CREATE TABLE IF NOT EXISTS event_logs (
        tx_hash          TEXT,
        log_index        INTEGER,
        block_number     INTEGER,
        contract_address TEXT,
        topic0           TEXT,
        topic1           TEXT,
        topic2           TEXT,
        data             TEXT,
        PRIMARY KEY (tx_hash, log_index)
    );

    CREATE INDEX IF NOT EXISTS idx_tx_from       ON transactions(from_address);
    CREATE INDEX IF NOT EXISTS idx_tx_date       ON transactions(block_date);
    CREATE INDEX IF NOT EXISTS idx_tt_contract   ON token_transfers(contract_address);
    CREATE INDEX IF NOT EXISTS idx_tt_date       ON token_transfers(block_date);
    CREATE INDEX IF NOT EXISTS idx_logs_topic0   ON event_logs(topic0);
""")
conn.commit()

# Verify
tables = [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()]
print(f" blockchain.db created at: {os.path.abspath(DB_PATH)}")
print(f" Tables: {', '.join(tables)}")

 blockchain.db created at: c:\Users\DELL\Desktop\MyProjects\week_5\blockchain.db
 Tables: event_logs, nft_transfers, token_transfers, transactions


In [4]:
# Quick API test
price = call("stats", "ethprice")
print(f" Etherscan API connected — ETH price: ${float(price['ethusd']):,.2f}")

 Etherscan API connected — ETH price: $2,129.17


### Ingest Live Ethereum Data from Etherscan

#### Normal transactions 

In [5]:
# Change this to any Ethereum wallet address you want to analyse
WALLET = "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045"  # vitalik.eth

def insert(sql, rows):
    conn.executemany(sql, rows)
    conn.commit()

tx_rows = []
for page in range(1, 6):
    batch = call("account", "txlist",
                 address=WALLET, startblock=0, endblock=99999999,
                 page=page, offset=100, sort="desc") or []
    for t in batch:
        gas_wei = int(t.get("gasUsed", 0)) * int(t.get("gasPrice", 0))
        tx_rows.append((
            t["hash"], int(t["blockNumber"]), ts_to_date(t["timeStamp"]),
            int(t["timeStamp"]), t["from"].lower(),
            (t.get("to") or "").lower() or None,
            wei_to_eth(t["value"]),
            int(t.get("gasUsed", 0)), int(t.get("gasPrice", 0)) / 1e9,
            gas_wei / 1e18,
            1 if t.get("isError") == "1" else 0,
        ))
    if len(batch) < 100: break
    time.sleep(0.22)

insert("""INSERT OR IGNORE INTO transactions
    (hash,block_number,block_date,block_timestamp,from_address,to_address,
     value_eth,gas_used,gas_price_gwei,gas_cost_eth,is_error)
    VALUES (?,?,?,?,?,?,?,?,?,?,?)""", tx_rows)
print(f"  {len(tx_rows)} transactions loaded")
time.sleep(0.3)

  500 transactions loaded


#### ERC-20 token transfers

In [6]:
tt_rows = []
for page in range(1, 4):
    batch = call("account", "tokentx",
                 address=WALLET, startblock=0, endblock=99999999,
                 page=page, offset=100, sort="desc") or []
    for i, t in enumerate(batch):
        dec = int(t.get("tokenDecimal", 18))
        tt_rows.append((
            t["hash"], i, ts_to_date(t["timeStamp"]), int(t["timeStamp"]),
            t["from"].lower(), t["to"].lower(),
            t["contractAddress"].lower(),
            t.get("tokenSymbol"), dec,
            int(t["value"]) / (10 ** dec) if t["value"] else None,
        ))
    if len(batch) < 100: break
    time.sleep(0.22)

insert("""INSERT OR IGNORE INTO token_transfers
    (hash,log_index,block_date,block_timestamp,from_address,to_address,
     contract_address,token_symbol,token_decimals,amount)
    VALUES (?,?,?,?,?,?,?,?,?,?)""", tt_rows)
print(f"{len(tt_rows)} token transfers loaded")
time.sleep(0.3)

300 token transfers loaded


#### ERC-721 NFT transfers

In [7]:
nft_rows = []
batch = call("account", "tokennfttx",
             address=WALLET, startblock=0, endblock=99999999,
             page=1, offset=100, sort="desc") or []
for i, t in enumerate(batch):
    nft_rows.append((
        t["hash"], i, ts_to_date(t["timeStamp"]),
        t["from"].lower(), t["to"].lower(),
        t["contractAddress"].lower(),
        t.get("tokenName"), t.get("tokenID"),
    ))
insert("""INSERT OR IGNORE INTO nft_transfers
    (hash,log_index,block_date,from_address,to_address,
     contract_address,token_name,token_id)
    VALUES (?,?,?,?,?,?,?,?)""", nft_rows)
print(f"{len(nft_rows)} NFT transfers loaded")
time.sleep(0.3)

100 NFT transfers loaded


#### Raw event logs (USDC Transfer events)

In [8]:
latest = int(call("proxy", "eth_blockNumber"), 16)
log_rows = []
batch = call("logs", "getLogs",
             address=CONTRACTS["USDC"], topic0=TRANSFER_TOPIC,
             fromBlock=latest - 300, toBlock="latest",
             page=1, offset=100) or []
for log in batch:
    topics = log.get("topics", [])
    log_rows.append((
        log["transactionHash"], int(log["logIndex"], 16),
        int(log["blockNumber"], 16), CONTRACTS["USDC"].lower(),
        topics[0] if len(topics) > 0 else None,
        topics[1] if len(topics) > 1 else None,
        topics[2] if len(topics) > 2 else None,
        log.get("data"),
    ))
insert("""INSERT OR IGNORE INTO event_logs
    (tx_hash,log_index,block_number,contract_address,topic0,topic1,topic2,data)
    VALUES (?,?,?,?,?,?,?,?)""", log_rows)
print(f"{len(log_rows)} event logs loaded")

100 event logs loaded


#### Row count summary

In [9]:
for tbl in ["transactions", "token_transfers", "nft_transfers", "event_logs"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<20}: {n:>5,} rows")

  transactions        :   500 rows
  token_transfers     :   300 rows
  nft_transfers       :   100 rows
  event_logs          :   100 rows


###  SQL Queries on Real Blockchain Data

In [10]:
import pandas as pd

def sql(query, title=""):
    """Run a SQL query and return a styled pandas DataFrame."""
    df = pd.read_sql_query(query, conn)
    if title:
        print(f"{title}")
    return df

#### Basic SELECT - top transactions by value


In [11]:
q1 = sql("""
    SELECT block_date,
           from_address,
           to_address,
           ROUND(value_eth, 4)      AS value_eth,
           ROUND(gas_cost_eth, 6)   AS gas_cost_eth,
           CASE WHEN is_error = 1 THEN 'failed' ELSE 'ok' END AS status
    FROM   transactions
    ORDER  BY value_eth DESC
    LIMIT  10
""", "Top 10 transactions by ETH value")
q1

Top 10 transactions by ETH value


,block_date,from_address,to_address,value_eth,gas_cost_eth,status
0,2025-11-26,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0x3c7779d27348017415a9184acadc0d62052841ab,1006.0000,0.000002,ok
1,2025-10-10,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0x3c7779d27348017415a9184acadc0d62052841ab,50.0000,0.000003,ok
2,2026-04-01,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0x3c7779d27348017415a9184acadc0d62052841ab,30.0000,0.000002,ok
3,2026-04-01,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0x3c7779d27348017415a9184acadc0d62052841ab,5.0000,0.000003,ok
4,2026-01-21,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0xfeb016d0d14ac0fa6d69199608b0776d007203b2,1.0000,0.000003,ok
5,2025-10-25,0xa94a1b69f4132bc835064fe5dd1e2c671d2b1833,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0.1000,0.000012,ok
6,2026-01-02,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0xc08e5c44a41ff606907697fe732ee36f28f37dee,0.0119,0.000002,ok
7,2026-02-28,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0xab5801a7d398351b8be11c439e05c5b3259aec9b,0.0100,0.000002,failed
8,2025-11-14,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0x97adc5f09e167127e40004b6a8b5742fc336ae28,0.0100,0.000003,ok
9,2025-11-07,0x3fdc4bae124d93536c55fc7f27d1fd0c6c54ec4d,0xd8da6bf26964af9d7eed9e03e53415d37aa96045,0.0063,0.000007,ok


#### GROUP BY + HAVING - daily activity summary

In [12]:
q2 = sql("""
    SELECT block_date,
           COUNT(*)                     AS tx_count,
           ROUND(SUM(value_eth), 4)     AS eth_moved,
           ROUND(AVG(gas_price_gwei),2) AS avg_gas_gwei,
           SUM(CASE WHEN is_error=1 THEN 1 ELSE 0 END) AS failed_txs
    FROM   transactions
    GROUP  BY block_date
    ORDER  BY block_date DESC
    LIMIT  14
""", "Daily activity summary (GROUP BY)")
q2

Daily activity summary (GROUP BY)


,block_date,tx_count,eth_moved,avg_gas_gwei,failed_txs
0,2026-04-01,12,35.0001,0.33,0
1,2026-03-31,1,0.0000,0.19,0
2,2026-03-27,10,0.0000,0.10,10
3,2026-03-25,1,0.0000,0.11,0
4,2026-03-19,1,0.0010,2.00,1
5,2026-03-18,1,0.0000,2.20,0
6,2026-03-16,2,0.0000,0.20,0
7,2026-03-14,1,0.0000,0.03,0
8,2026-03-13,5,0.0001,0.99,0
9,2026-03-12,2,0.0000,0.05,0


#### JOIN - transactions enriched with token transfers

In [13]:
q3 = sql("""
    SELECT t.block_date,
           t.hash,
           ROUND(t.value_eth, 4)    AS eth_value,
           tt.token_symbol,
           ROUND(tt.amount, 2)      AS token_amount
    FROM       transactions   t
    INNER JOIN token_transfers tt ON t.hash = tt.hash
    ORDER BY   t.block_date DESC
    LIMIT  10
""", "JOIN: transactions + token transfers")
q3

JOIN: transactions + token transfers


,block_date,hash,eth_value,token_symbol,token_amount
0,2026-04-01,0x401201328cf7dff2bcb891afb6965ca81d4683cc7d62...,0.0,USDC,70000.0
1,2026-02-11,0x751120f892f594f6eae405f1c5b616285b7b731a1375...,0.0,USDC,2973.0


#### Window function - cumulative ETH moved + daily rank

In [14]:
q4 = sql("""
    SELECT block_date,
           ROUND(SUM(value_eth), 4)  AS daily_eth,
           ROUND(SUM(SUM(value_eth)) OVER (ORDER BY block_date), 4) AS cumulative_eth,
           RANK() OVER (ORDER BY SUM(value_eth) DESC)               AS rank_by_volume
    FROM   transactions
    WHERE  is_error = 0
    GROUP  BY block_date
    ORDER  BY block_date DESC
    LIMIT  14
""", "Window function: cumulative ETH + daily rank")
q4

Window function: cumulative ETH + daily rank


,block_date,daily_eth,cumulative_eth,rank_by_volume
0,2026-04-01,35.0001,1092.1763,3
1,2026-03-31,0.0000,1057.1763,112
2,2026-03-25,0.0000,1057.1762,139
3,2026-03-18,0.0000,1057.1762,117
4,2026-03-16,0.0000,1057.1762,132
5,2026-03-14,0.0000,1057.1762,119
6,2026-03-13,0.0001,1057.1762,47
7,2026-03-12,0.0000,1057.1761,116
8,2026-03-09,0.0000,1057.1761,110
9,2026-03-08,0.0051,1057.1761,10


#### Raw event log decoding - USDC Transfer events


In [15]:
q5 = sql("""
    SELECT block_number,
           tx_hash,
           '0x' || SUBSTR(topic1, LENGTH(topic1)-39) AS from_address,
           '0x' || SUBSTR(topic2, LENGTH(topic2)-39) AS to_address,
           ROUND(CAST(('0x' || LTRIM(REPLACE(data,'0x',''), '0')) AS INTEGER)
                 / 1000000.0, 2)                      AS usdc_amount
    FROM   event_logs
    WHERE  topic0 = '0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef'
      AND  data NOT IN ('0x', '')
    ORDER  BY block_number DESC
    LIMIT  10
""", "Raw log decoding: USDC Transfer events")
q5

Raw log decoding: USDC Transfer events


,block_number,tx_hash,from_address,to_address,usdc_amount
0,24786549,0x8470c3d0cae4c2032ecad2278dc0acd8e70f7976426b...,0xf763bb342eb3d23c02ccb86312422fe0c1c17e94,0x111111125421ca6dc452d289314280a0f8842a65,0.0
1,24786549,0x8470c3d0cae4c2032ecad2278dc0acd8e70f7976426b...,0x00000000000014aa86c5d3c41765bb24e11bd701,0x111111125421ca6dc452d289314280a0f8842a65,0.0
2,24786549,0x8470c3d0cae4c2032ecad2278dc0acd8e70f7976426b...,0x000000000004444c5dc75cb358380d2e3de08a90,0x111111125421ca6dc452d289314280a0f8842a65,0.0
3,24786549,0x8470c3d0cae4c2032ecad2278dc0acd8e70f7976426b...,0x111111125421ca6dc452d289314280a0f8842a65,0x00000000008d5f1200332af8a6813cb8377b5bfd,0.0
4,24786549,0x8470c3d0cae4c2032ecad2278dc0acd8e70f7976426b...,0x00000000008d5f1200332af8a6813cb8377b5bfd,0xf6d58ace340b49ac800523b0877362e273a76f5a,0.0
5,24786549,0x4a53886d1f6ef8ab0fc97de5976da4691a424adecc4d...,0x00000000000014aa86c5d3c41765bb24e11bd701,0x63242a4ea82847b20e506b63b0e2e2eff0cc6cb0,0.0
6,24786549,0x4a53886d1f6ef8ab0fc97de5976da4691a424adecc4d...,0xe3d41d19564922c9952f692c5dd0563030f5f2ef,0x63242a4ea82847b20e506b63b0e2e2eff0cc6cb0,0.0
7,24786549,0x4a53886d1f6ef8ab0fc97de5976da4691a424adecc4d...,0x1445f32d1a74872ba41f3d8cf4022e9996120b31,0x63242a4ea82847b20e506b63b0e2e2eff0cc6cb0,0.0
8,24786549,0x4a53886d1f6ef8ab0fc97de5976da4691a424adecc4d...,0x63242a4ea82847b20e506b63b0e2e2eff0cc6cb0,0x62b599de152f8e689088f0011e39824858bc1ef1,0.0
9,24786549,0x0a4f8829579316c24f38332f710b742ab924b0edd39a...,0x4f493b7de8aac7d55f71853688b1f7c8f0243c85,0x1445f32d1a74872ba41f3d8cf4022e9996120b31,0.0


### Pandas Analysis & CSV Export

In [16]:
import pandas as pd

# Load core tables into DataFrames 
df_tx = pd.read_sql_query("SELECT * FROM transactions", conn)
df_tt = pd.read_sql_query("SELECT * FROM token_transfers", conn)

df_tx["block_date"] = pd.to_datetime(df_tx["block_date"])
df_tt["block_date"] = pd.to_datetime(df_tt["block_date"])

print(f"Loaded {len(df_tx):,} transactions and {len(df_tt):,} token transfers\n")

Loaded 500 transactions and 300 token transfers



#### Gas spending summary 

In [17]:
gas_summary = df_tx.agg(
    total_gas_eth  = ("gas_cost_eth", "sum"),
    avg_gas_gwei   = ("gas_price_gwei", "mean"),
    max_gas_gwei   = ("gas_price_gwei", "max"),
    min_gas_gwei   = ("gas_price_gwei", "min"),
).round(6)
print("Gas Spending Summary")
print(gas_summary.to_string())

Gas Spending Summary
               gas_cost_eth  gas_price_gwei
total_gas_eth      0.013778             NaN
avg_gas_gwei            NaN        0.834033
max_gas_gwei            NaN       20.000000
min_gas_gwei            NaN        0.024169


#### Monthly cohort 

In [18]:
monthly = (
    df_tx.assign(month=df_tx["block_date"].dt.to_period("M"))
         .groupby("month")
         .agg(
             tx_count       = ("hash",         "count"),
             eth_moved      = ("value_eth",     "sum"),
             gas_spent_eth  = ("gas_cost_eth",  "sum"),
             failed         = ("is_error",      "sum"),
         )
         .round(6)
         .sort_index(ascending=False)
)
print("\nMonthly Cohort")
monthly


Monthly Cohort


,tx_count,eth_moved,gas_spent_eth,failed
month,,,,
2026-04,12,35.000095,0.000089,0
2026-03,38,0.006355,0.000592,11
2026-02,48,0.024700,0.000444,5
2026-01,73,1.016083,0.004545,2
2025-12,132,0.004091,0.002470,3
2025-11,57,1006.022042,0.000854,1
2025-10,113,50.108960,0.002939,8
2025-09,27,0.005532,0.001846,1


#### Export to CSV

In [19]:
df_tx.to_csv("transactions.csv", index=False)
monthly.to_csv("monthly_cohort.csv")